# Failure Recovery — fine-tune quy mô nhỏ

Tái hiện nhánh **static evaluation (FaultEval)** của paper VLA-FixBench: dạy một model nhỏ phân loại **loại lỗi** (nominal / perception / planning / control) và **mức nặng** (0–3), rồi đo độ rớt phong độ trên slice **OOD** (kiểu FLARE: ID vs OOD).

- Dataset thu gọn: 3000 train / 600 val / 600 test (+600 OOD), sinh bằng quy tắc lấy từ taxonomy 2 paper.
- Model nhỏ: MLP ~6k tham số, 15 epoch. CPU Colab chạy ~1–2 phút, có GPU càng nhanh.
- Không cần cài gì thêm: torch + numpy + matplotlib có sẵn trên Colab.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Dữ liệu fault tổng hợp

Mỗi mẫu là một vector quan sát lúc robot thao tác: độ lệch tay máy (2 chiều), khoảng cách tới vật, độ trễ điều khiển, độ mờ ảnh. Nhãn loại lỗi suy từ đúng 3 nhóm của VLA-FixBench: mờ ảnh → perception, trễ lớn → planning, lệch tay máy → control.

In [ ]:
def make_fault_data(n, ood=False, seed=0):
    rng = np.random.default_rng(seed)
    s = 2.0 if ood else 1.0
    ee = rng.uniform(-1, 1, (n, 2)) * s
    dist = np.abs(rng.normal(0.4, 0.5, (n, 1)))
    delay = rng.uniform(0, 1, (n, 1))
    blur = rng.uniform(0, 1, (n, 1))
    X = np.hstack([ee, dist, delay, blur]).astype(np.float32)

    ftype = np.zeros(n, dtype=np.int64)
    mag = np.abs(ee).max(axis=1)
    perc = blur[:, 0] > 0.75
    plan = (~perc) & (delay[:, 0] > 0.8)
    ctrl = (~perc) & (~plan) & ((mag > 0.6) | (dist[:, 0] > 1.0))
    ftype[perc], ftype[plan], ftype[ctrl] = 1, 2, 3

    sev = np.clip((mag * 2 + blur[:, 0] + delay[:, 0]).astype(int), 0, 3)
    sev[ftype == 0] = 0
    return X, ftype, sev

Xtr, ytr_t, ytr_s = make_fault_data(3000, seed=1)
Xva, yva_t, yva_s = make_fault_data(600, seed=2)
Xte, yte_t, yte_s = make_fault_data(600, seed=3)
Xood, yood_t, yood_s = make_fault_data(600, ood=True, seed=4)

names = ["nominal", "perception", "planning", "control"]
for name, y in [("train", ytr_t), ("test", yte_t), ("ood", yood_t)]:
    u, c = np.unique(y, return_counts=True)
    print(name, {names[i]: int(c[list(u).index(i)]) if i in u else 0 for i in range(4)})

## 2. Model nhỏ, train ít epoch

MLP 2 đầu ra: một đầu đoán loại lỗi (FTC của FaultEval), một đầu đoán mức nặng (FSR).

In [ ]:
class FaultNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(5, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.ht = nn.Linear(32, 4)
        self.hs = nn.Linear(32, 4)
    def forward(self, x):
        h = self.body(x)
        return self.ht(h), self.hs(h)

print("params:", sum(p.numel() for p in FaultNet().parameters()))

Xtr_t = torch.from_numpy(Xtr)
ytr1, ytr2 = torch.from_numpy(ytr_t), torch.from_numpy(ytr_s)
Xva_t = torch.from_numpy(Xva)
yva1, yva2 = torch.from_numpy(yva_t), torch.from_numpy(yva_s)

model = FaultNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
ce = nn.CrossEntropyLoss()
hist = []
for ep in range(1, 16):
    model.train()
    perm = torch.randperm(len(Xtr_t))
    tot, nb = 0.0, 0
    for i in range(0, len(Xtr_t), 128):
        b = perm[i:i + 128]
        lt, ls = model(Xtr_t[b].to(device))
        loss = ce(lt, ytr1[b].to(device)) + ce(ls, ytr2[b].to(device))
        opt.zero_grad(); loss.backward(); opt.step()
        tot += float(loss) * len(b); nb += len(b)
    model.eval()
    with torch.no_grad():
        pt, ps = model(Xva_t.to(device))
        acc = ((pt.argmax(1).cpu() == yva1).float().mean() + (ps.argmax(1).cpu() == yva2).float().mean()).item() / 2
    hist.append((tot / nb, acc))
    print(f"epoch {ep:2d}  loss {tot / nb:.4f}  val_acc {acc:.4f}")

import matplotlib.pyplot as plt
lt, la = zip(*hist)
plt.figure(figsize=(8, 3.5))
plt.subplot(1, 2, 1); plt.plot(lt); plt.title("train loss"); plt.xlabel("epoch")
plt.subplot(1, 2, 2); plt.plot(la); plt.title("val acc (trung binh 2 dau)"); plt.xlabel("epoch")
plt.tight_layout(); plt.show()

## 3. Đánh giá: test thường + slice OOD

Đo FTC accuracy (so với baseline đoán lớp đông nhất) và severity accuracy, riêng trên test ID và test OOD — đúng kiểu FLARE tách ID/OOD.

In [ ]:
@torch.no_grad()
def evaluate(X, yt, ys):
    model.eval()
    p1, p2 = model(torch.from_numpy(X).to(device))
    p1, p2 = p1.argmax(1).cpu().numpy(), p2.argmax(1).cpu().numpy()
    return (p1 == yt).mean(), (p2 == ys).mean(), p1

for name, D in [("test", (Xte, yte_t, yte_s)), ("ood", (Xood, yood_t, yood_s))]:
    a1, a2, _ = evaluate(*D)
    maj = float(np.bincount(D[1]).max() / len(D[1]))
    print(f"{name}: FTC_acc={a1:.3f} (majority={maj:.3f})  SEV_acc={a2:.3f}")

_, _, p = evaluate(Xte, yte_t, yte_s)
cm = np.zeros((4, 4), dtype=int)
for t, q in zip(yte_t, p):
    cm[t, q] += 1
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4), names, rotation=20)
ax.set_yticks(range(4), names)
ax.set_xlabel("du doan"); ax.set_ylabel("that")
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout(); plt.show()

## 4. Đọc kết quả

- FTC trên test ID phải vượt xa baseline majority — model học được ranh giới 3 nhóm lỗi, tương ứng cột FTC trong Table 1 của paper 2.
- Slice OOD (nhiễu gấp đôi) rớt vài điểm: cùng hiện tượng paper 2 ghi nhận — model phân biệt tốt trong phân phối train nhưng kém khi dịch phân phối, và cũng là lý do FLARE phải sinh thêm dữ liệu perturbation.
- Limitation của bài này: nhãn suy từ quy tắc nên model chỉ học lại quy tắc đó; muốn sát paper hơn thì thay dữ liệu tổng hợp bằng log lỗi thật (LIBERO / robot) rồi fine-tune tiếp từ checkpoint này.

Chép 4 số (FTC test, FTC OOD, SEV test, SEV OOD) gửi lại để điền vào mục 5 của báo cáo.